# Kafka Consumer and Energy Prediction Visualisation

This notebook consumes prediction outputs from Kafka and creates visualisations for real-time building energy monitoring. It combines prediction streams with actual metered data to evaluate daily shortfall and excess energy by site.

## Overview

The consumer visualisation notebook subscribes to Kafka topics created by the streaming prediction pipeline. It produces three analysis views: 6-hour building-level prediction trends, daily site-level energy summaries, and predicted-vs-actual shortfall/excess analysis.

In [ ]:
# ============================================
# CONSUMER VISUALISATION: IMPORT LIBRARIES
# ============================================

# Data manipulation
import pandas as pd
import numpy as np

# Kafka consumer
from kafka import KafkaConsumer

# JSON handling
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

# Date/time
from datetime import datetime
import time

# Display settings for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)

# Matplotlib settings for better plots
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.style.use('default')

print("All libraries imported successfully!")

## Configuration Constants

This section defines input file paths and Kafka topic names. The topic names must match the streaming prediction notebook outputs: `predictions_raw`, `predictions_6h`, and `predictions_daily`.

In [ ]:
# ============================================
# CONSUMER VISUALISATION: CONFIGURATION
# ============================================

# Kafka broker address (same as producer notebook and Spark streaming notebook)
KAFKA_BROKER = 'host.docker.internal:9092'

# Kafka topic names (created by Spark streaming notebook Kafka publishing stage)
KAFKA_TOPIC_RAW = 'predictions_raw'         # From raw prediction stream (optional, not required for plots)
KAFKA_TOPIC_6H = 'predictions_6h'           # From 6-hour aggregation stream (for Plot 1)
KAFKA_TOPIC_DAILY = 'predictions_daily'     # From daily aggregation stream (for Plot 2 & 3)

# File paths (update if your files are in different locations)
METERS_CSV_PATH = 'new_meters.csv'
BUILDINGS_CSV_PATH = 'new_building_information.csv'

print("Configuration loaded:")
print(f"  Kafka Broker: {KAFKA_BROKER}")
print(f"  Topics:")
print(f"    - Raw predictions:  {KAFKA_TOPIC_RAW}")
print(f"    - 6h aggregations:  {KAFKA_TOPIC_6H}")
print(f"    - Daily aggregations: {KAFKA_TOPIC_DAILY}")
print(f"  Files:")
print(f"    - Meters: {METERS_CSV_PATH}")
print(f"    - Buildings: {BUILDINGS_CSV_PATH}")

## Load Actual Meter Data

The new meter CSV contains actual energy consumption readings. These values are used as ground truth for comparing model predictions against measured daily energy consumption.

## Actual Meter Data Preparation

Meter readings are loaded into pandas, parsed into timestamps, and later aggregated by site and day. This prepares the actual-consumption baseline for the prediction error analysis.

In [ ]:
# ============================================
# CONSUMER VISUALISATION.1: LOAD NEW METERS CSV
# ============================================

# Load meter data from CSV
meters_df = pd.read_csv(METERS_CSV_PATH)

print("="*70)
print("METERS DATA LOADED")
print("="*70)
print(f"Total records: {len(meters_df):,}")

# Display structure
print(f"\nColumns: {list(meters_df.columns)}")
print(f"\nFirst 5 records:")
print(meters_df.head())

print(f"\nData types:")
print(meters_df.dtypes)

# Check for missing values
print(f"\nMissing values:")
print(meters_df.isnull().sum())

# Basic statistics
print(f"\nBasic statistics:")
print(meters_df['value'].describe())

print("\nConsumer visualisation.1 Completed!")

## Kafka Message Consumer Helper

This helper function reads JSON messages from a selected Kafka topic and converts them into a pandas DataFrame. It supports quick local analysis and visualisation of streaming results.

In [ ]:
# ============================================
# HELPER FUNCTION: CONSUME KAFKA MESSAGES
# ============================================

def consume_kafka_messages(topic, num_messages=100, timeout_ms=30000):
    """
    Consume messages from Kafka topic and return as pandas DataFrame.
    
    Args:
        topic (str): Kafka topic name
        num_messages (int): Maximum number of messages to consume
        timeout_ms (int): Timeout in milliseconds (30 seconds default)
        
    Returns:
        pd.DataFrame: DataFrame containing consumed messages
        
    Example:
        df = consume_kafka_messages('predictions_6h', num_messages=50)
    """
    print(f"\n{'='*70}")
    print(f"Consuming from topic: {topic}")
    print(f"{'='*70}")
    
    try:
        # Create Kafka consumer
        consumer = KafkaConsumer(
            topic,
            bootstrap_servers=KAFKA_BROKER,
            auto_offset_reset='earliest',      # Start from beginning
            enable_auto_commit=False,          # Don't commit offsets (for testing)
            consumer_timeout_ms=timeout_ms,    # Timeout after N milliseconds
            value_deserializer=lambda m: json.loads(m.decode('utf-8'))  # Parse JSON
        )
        
        messages = []
        count = 0
        
        # Consume messages
        for message in consumer:
            messages.append(message.value)
            count += 1
            
            # Progress indicator every 10 messages
            if count % 10 == 0:
                print(f"  Consumed {count} messages...")
            
            # Stop after reaching num_messages
            if count >= num_messages:
                break
        
        # Close consumer
        consumer.close()
        
        print(f"Total messages consumed: {count}")
        
        # Convert to DataFrame
        if count > 0:
            df = pd.DataFrame(messages)
            print(f"DataFrame created with {len(df)} rows and {len(df.columns)} columns")
            return df
        else:
            print("No messages found in topic")
            print("   Possible reasons:")
            print("   1. Spark streaming notebook Kafka publishing stage is not running")
            print("   2. No data has been produced yet")
            print("   3. Topic name is incorrect")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"Error consuming from Kafka: {e}")
        print(f"   Check:")
        print(f"   1. Kafka broker is accessible at {KAFKA_BROKER}")
        print(f"   2. Topic '{topic}' exists")
        print(f"   3. Spark streaming notebook Kafka publishing stage is producing data")
        return pd.DataFrame()

print("Helper function defined successfully")

## Create Visualisations from Streaming Outputs

The first two plots visualise data published by the streaming pipeline: 6-hour building-level aggregations and daily site-level aggregations.

## Streaming Visualisation Plan

## Plot 1: 6-Hour Building-Level Energy Consumption

This plot uses the `predictions_6h` Kafka topic to show energy-consumption trends across 6-hour windows for selected buildings.

In [ ]:
# ============================================
# CONSUMER VISUALISATION.2a: CONSUME 6-HOUR AGGREGATION DATA
# ============================================
print("\n" + "="*70)
print("CONSUMER VISUALISATION.2a: CONSUMING 6-HOUR AGGREGATION DATA")
print("="*70)

# Define Kafka topic for 6-hour aggregations
KAFKA_TOPIC_6H = 'predictions_6h'

# Consume messages from Kafka
# Increase num_messages if you want more historical data
df_6h = consume_kafka_messages(KAFKA_TOPIC_6H, num_messages=300, timeout_ms=30000)

# Verify data was received
if not df_6h.empty:
    print(f"\n 6-HOUR DATA SUCCESSFULLY LOADED")
    print(f"{'='*70}")
    
    # Display dataset shape
    print(f"\n DATASET OVERVIEW:")
    print(f"   Shape: {df_6h.shape[0]} rows × {df_6h.shape[1]} columns")
    print(f"   Columns: {list(df_6h.columns)}")
    
    # Display first few records
    print(f"\n SAMPLE DATA (First 5 records):")
    print(df_6h.head(5).to_string(index=False))
    
    # Data type information
    print(f"\n DATA TYPES:")
    for col, dtype in df_6h.dtypes.items():
        print(f"   {col:20s} → {dtype}")
    
    # Convert timestamp if needed
    if 'window_start' in df_6h.columns:
        if df_6h['window_start'].dtype == 'object':
            df_6h['window_start'] = pd.to_datetime(df_6h['window_start'])
            print(f"\n✓ Converted 'window_start' to datetime format")
    
    # Display statistics
    print(f"\n STATISTICS:")
    print(f"   Unique buildings: {df_6h['building_id'].nunique()}")
    print(f"   Unique sites: {df_6h['site_id'].nunique() if 'site_id' in df_6h.columns else 'N/A'}")
    print(f"   Time range: {df_6h['window_start'].min()} to {df_6h['window_start'].max()}")
    print(f"   Total energy: {df_6h['total_energy_6h'].sum():,.2f} kWh")
    print(f"   Average energy: {df_6h['total_energy_6h'].mean():,.2f} kWh per 6-hour window")
    print(f"   Max energy: {df_6h['total_energy_6h'].max():,.2f} kWh")
    print(f"   Min energy: {df_6h['total_energy_6h'].min():,.2f} kWh")
    
    print("\n" + "="*70)
    print(" 6-HOUR DATA READY FOR VISUALIZATION (PLOT 1)")
    print("="*70)
    
else:
    print("\n NO 6-HOUR DATA AVAILABLE")
    print("="*70)
    print("🔧 TROUBLESHOOTING STEPS:")
    print("   1. Ensure Spark streaming notebook, 6-hour aggregation stream (query_8b) is running")
    print("   2. Wait 10-15 seconds for data to be produced to Kafka")
    print("   3. Verify Kafka topic name matches: 'predictions_6h'")
    print("   4. Check that the 6-hour Parquet sink is writing to Parquet successfully")
    print("="*70)

## Plot 1 Design Notes

A line plot is used to show changes over time. The chart focuses on the highest-consumption buildings to keep the visualisation readable.

In [ ]:
# ============================================
# PLOT 1: 6-HOUR ENERGY CONSUMPTION BY BUILDING
# ============================================
print("\n" + "="*70)
print("CREATING PLOT 1: 6-HOUR ENERGY CONSUMPTION")
print("="*70)

if not df_6h.empty:
    
    # ----------------------------------------
    # DATA PREPARATION
    # ----------------------------------------
    # Ensure window_start is datetime for proper time-series plotting
    df_6h['window_start'] = pd.to_datetime(df_6h['window_start'])
    
    # Identify top 8 buildings by total energy consumption
    # This focuses the visualization on the most significant consumers
    top_buildings = (df_6h.groupby('building_id')['total_energy_6h']
                     .sum()                    # Sum energy across all time windows
                     .nlargest(8)              # Get top 8
                     .index)                   # Extract building IDs
    
    print(f"Top 8 buildings by total consumption: {list(top_buildings)}")
    
    # ----------------------------------------
    # FIGURE SETUP
    # ----------------------------------------
    fig, ax = plt.subplots(figsize=(18, 10), dpi=120)
    
    # Generate distinct colors for each building
    colors = plt.cm.tab10(range(len(top_buildings)))
    
    # ----------------------------------------
    # PLOT EACH BUILDING'S TIME SERIES
    # ----------------------------------------
    for idx, building_id in enumerate(top_buildings):
        # Filter data for current building
        building_data = (df_6h[df_6h['building_id'] == building_id]
                        .sort_values('window_start'))  # Sort by time for proper line plot
        
        if len(building_data) > 0:
            # Plot line with markers
            ax.plot(
                building_data['window_start'],      # X-axis: time
                building_data['total_energy_6h'],   # Y-axis: energy
                marker='o',                          # Circle markers at each data point
                markersize=8,                        # Marker size
                linewidth=2.5,                       # Line thickness
                label=f'Building {building_id}',    # Legend label
                color=colors[idx],                   # Unique color
                alpha=0.85                           # Slight transparency
            )
    
    # ----------------------------------------
    # STYLING AND FORMATTING
    # ----------------------------------------
    # Axis labels
    ax.set_xlabel('Time Window (6-Hour Intervals)', 
                  fontsize=14, fontweight='bold', labelpad=15)
    ax.set_ylabel('Energy Consumption (kWh)', 
                  fontsize=14, fontweight='bold', labelpad=15)
    
    # Title with descriptive subtitle
    ax.set_title('6-Hour Energy Consumption by Building\nTop 8 Buildings - Real-Time Streaming Data from Kafka', 
                 fontsize=16, fontweight='bold', pad=25)
    
    # Legend configuration
    ax.legend(
        loc='upper left',           # Position
        frameon=True,                # Show frame
        shadow=True,                 # Add shadow effect
        fancybox=True,              # Rounded corners
        fontsize=11,                 # Text size
        title='Building ID',         # Legend title
        title_fontsize=12            # Title size
    )
    
    # Grid for easier reading
    ax.grid(True, alpha=0.3, linestyle='--', linewidth=1)
    ax.set_facecolor('#f8f9fa')      # Light gray background
    ax.set_axisbelow(True)           # Grid behind data
    
    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha='right')
    
    # Format y-axis with thousand separators
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    
    # ----------------------------------------
    # LIVE DATA INDICATOR
    # ----------------------------------------
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ax.text(0.02, 0.98, 
            f'** LIVE STREAMING DATA **\nUpdated: {current_time}\nRecords: {len(df_6h)}',
            transform=ax.transAxes,      # Use axes coordinates (0-1)
            fontsize=11, 
            verticalalignment='top',
            bbox=dict(
                boxstyle='round,pad=0.7', 
                facecolor='red', 
                alpha=0.35,
                edgecolor='red',
                linewidth=2
            ))
    
    # ----------------------------------------
    # STATISTICS BOX
    # ----------------------------------------
    total_energy = df_6h['total_energy_6h'].sum()
    avg_energy = df_6h['total_energy_6h'].mean()
    max_energy = df_6h['total_energy_6h'].max()
    
    stats_text = f'STATISTICS\n'
    stats_text += f'─────────────────\n'
    stats_text += f'Total: {total_energy:,.0f} kWh\n'
    stats_text += f'Average: {avg_energy:,.0f} kWh\n'
    stats_text += f'Peak: {max_energy:,.0f} kWh'
    
    ax.text(0.98, 0.98, stats_text,
            transform=ax.transAxes,
            fontsize=11,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(
                boxstyle='round,pad=0.7',
                facecolor='#e3f2fd',
                alpha=0.95,
                edgecolor='#1976d2',
                linewidth=2
            ),
            family='monospace')          # Monospace for aligned numbers
    
    # ----------------------------------------
    # FINALIZE AND DISPLAY
    # ----------------------------------------
    plt.tight_layout()
    plt.savefig('plot1_6hour_energy.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*70)
    print("PLOT 1 CREATED SUCCESSFULLY!")
    print("="*70)
    print(f"Saved as: plot1_6hour_energy.png")
    print(f"Visualized {len(df_6h)} records from Kafka")
    print(f"Displayed top 8 buildings out of {df_6h['building_id'].nunique()} total")
    print("="*70)
    
else:
    print("\n" + "="*70)
    print("CANNOT CREATE PLOT 1 - NO 6-HOUR DATA AVAILABLE")
    print("="*70)

## Plot 2: Daily Energy Consumption by Site

This plot uses the `predictions_daily` Kafka topic to compare predicted daily energy consumption across sites and dates.

In [ ]:
# ============================================
# CONSUMER VISUALISATION.2b: CONSUME DAILY AGGREGATION DATA
# ============================================
print("\n" + "="*70)
print("CONSUMER VISUALISATION.2b: CONSUMING DAILY AGGREGATION DATA")
print("="*70)

# Define Kafka topic for daily aggregations
KAFKA_TOPIC_DAILY = 'predictions_daily'

# Consume messages from Kafka
df_daily = consume_kafka_messages(KAFKA_TOPIC_DAILY, num_messages=150, timeout_ms=30000)

# Verify data was received
if not df_daily.empty:
    print(f"\n DAILY DATA SUCCESSFULLY LOADED")
    print(f"{'='*70}")
    
    # Display dataset overview
    print(f"\n DATASET OVERVIEW:")
    print(f"   Shape: {df_daily.shape[0]} rows × {df_daily.shape[1]} columns")
    print(f"   Columns: {list(df_daily.columns)}")
    
    # Display first few records
    print(f"\n SAMPLE DATA (First 5 records):")
    print(df_daily.head(5).to_string(index=False))
    
    # Data type information
    print(f"\n DATA TYPES:")
    for col, dtype in df_daily.dtypes.items():
        print(f"   {col:20s} → {dtype}")
    
    # Convert day column if needed
    if 'day' in df_daily.columns:
        if df_daily['day'].dtype == 'object':
            df_daily['day'] = pd.to_datetime(df_daily['day'])
            print(f"\n✓ Converted 'day' to datetime format")
    
    # Display statistics
    print(f"\n STATISTICS:")
    print(f"   Unique sites: {df_daily['site_id'].nunique()}")
    print(f"   Unique days: {df_daily['day'].nunique()}")
    print(f"   Date range: {df_daily['day'].min()} to {df_daily['day'].max()}")
    print(f"   Total energy: {df_daily['total_energy_daily'].sum():,.2f} kWh")
    print(f"   Average per site-day: {df_daily['total_energy_daily'].mean():,.2f} kWh")
    print(f"   Max daily consumption: {df_daily['total_energy_daily'].max():,.2f} kWh")
    print(f"   Min daily consumption: {df_daily['total_energy_daily'].min():,.2f} kWh")
    
    print("\n" + "="*70)
    print(" DAILY DATA READY FOR VISUALIZATION (PLOT 2 & 3)")
    print("="*70)
    
else:
    print("\n NO DAILY DATA AVAILABLE")
    print("="*70)
    print(" TROUBLESHOOTING STEPS:")
    print("   1. Ensure Spark streaming notebook, daily aggregation stream (query_8c) is running")
    print("   2. Wait 15-20 seconds for data to be produced to Kafka")
    print("   3. Verify Kafka topic name matches: 'predictions_daily'")
    print("   4. Check that the daily Parquet sink is writing to Parquet successfully")
    print("="*70)

## Plot 2 Design Notes

A horizontal bar chart improves readability when comparing many site-date combinations. Labels and summary statistics are added to make the output suitable for reporting.

In [ ]:
# ============================================
# PLOT 2: DAILY ENERGY CONSUMPTION BY SITE
# Ultimate Horizontal Bar Chart with Maximum Clarity
# ============================================
print("\n" + "="*70)
print("CREATING PLOT 2: DAILY ENERGY CONSUMPTION BY SITE")
print("="*70)

if not df_daily.empty:
    
    # ----------------------------------------
    # DATA PREPARATION
    # ----------------------------------------
    # Create working copy to avoid modifying original
    plot_data = df_daily.copy()
    
    # Create combined label: Date + Site for clear identification
    # Format: "YYYY-MM-DD\nSite X" (two lines for readability)
    plot_data['date_site'] = (plot_data['day'].astype(str) + '\nSite ' + 
                              plot_data['site_id'].astype(str))
    
    # Sort by date and site for logical ordering
    plot_data = plot_data.sort_values(['day', 'site_id'])
    
    print(f"Preparing visualization for {len(plot_data)} site-day combinations")
    
    # ----------------------------------------
    # FIGURE SETUP
    # ----------------------------------------
    # Dynamic height based on number of bars (minimum 14 inches)
    fig_height = max(14, len(plot_data) * 0.5)
    fig, ax = plt.subplots(figsize=(20, fig_height), dpi=120)
    
    # ----------------------------------------
    # COLOR MAPPING
    # ----------------------------------------
    # Create consistent colors for each site across all dates
    unique_sites = sorted(plot_data['site_id'].unique())
    colors_map = plt.cm.Spectral(np.linspace(0.1, 0.9, len(unique_sites)))
    site_colors = {site: colors_map[i] for i, site in enumerate(unique_sites)}
    bar_colors = [site_colors[site] for site in plot_data['site_id']]
    
    print(f"Generated color scheme for {len(unique_sites)} sites")
    
    # ----------------------------------------
    # CREATE HORIZONTAL BARS
    # ----------------------------------------
    bars = ax.barh(
        range(len(plot_data)),              # Y positions (0, 1, 2, ...)
        plot_data['total_energy_daily'],    # Bar lengths (energy values)
        color=bar_colors,                    # Colors mapped to sites
        alpha=0.9,                           # Slight transparency
        edgecolor='black',                   # Black borders for clarity
        linewidth=2.5,                       # Thick borders
        height=0.75                          # Bar height (spacing)
    )
    
    # ----------------------------------------
    # Y-AXIS LABELS (DATE + SITE)
    # ----------------------------------------
    ax.set_yticks(range(len(plot_data)))
    ax.set_yticklabels(plot_data['date_site'], fontsize=13, fontweight='bold')
    
    # ----------------------------------------
    # DATA LABELS ON BARS
    # ----------------------------------------
    max_value = plot_data['total_energy_daily'].max()
    
    for i, (bar, value, site, date) in enumerate(zip(
        bars, 
        plot_data['total_energy_daily'], 
        plot_data['site_id'],
        plot_data['day']
    )):
        # Value label at end of bar (outside)
        ax.text(value, bar.get_y() + bar.get_height()/2,
               f'  {value:,.0f} kWh',              # Format with commas
               ha='left',                           # Align left (outside bar)
               va='center',                         # Vertically centered
               fontsize=12, 
               fontweight='bold',
               bbox=dict(
                   boxstyle='round,pad=0.5', 
                   facecolor='white', 
                   alpha=0.95,
                   edgecolor='black',
                   linewidth=1.5
               ))
        
        # Site ID label inside bar (if bar is long enough)
        if value > max_value * 0.12:  # Only show if bar > 12% of max
            ax.text(value * 0.5,                       # Middle of bar
                   bar.get_y() + bar.get_height()/2,
                   f'SITE {site}',
                   ha='center', 
                   va='center', 
                   fontsize=14, 
                   fontweight='bold',
                   color='white',                      # White text
                   bbox=dict(
                       boxstyle='round,pad=0.6', 
                       facecolor='black',              # Black background
                       alpha=0.85,
                       edgecolor='white',
                       linewidth=2
                   ))
    
    # ----------------------------------------
    # AXIS LABELS AND TITLE
    # ----------------------------------------
    ax.set_xlabel('Daily Energy Consumption (kWh)', 
                  fontsize=18, fontweight='bold', labelpad=20)
    ax.set_ylabel('Date / Site ID', 
                  fontsize=18, fontweight='bold', labelpad=20)
    ax.set_title('Daily Energy Consumption by Site\nMulti-Site Performance Analysis - Real-Time Streaming Data', 
                fontsize=22, fontweight='bold', pad=30)
    
    # ----------------------------------------
    # GRID AND STYLING
    # ----------------------------------------
    ax.grid(True, alpha=0.4, axis='x', linestyle='--', linewidth=1.5)
    ax.set_facecolor('#f8f9fa')          # Light background
    ax.set_axisbelow(True)               # Grid behind bars
    
    # Format x-axis with thousand separators
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    ax.tick_params(axis='x', labelsize=14, width=2, length=6)
    ax.tick_params(axis='y', width=2, length=6)
    
    # ----------------------------------------
    # LEGEND
    # ----------------------------------------
    legend_elements = [
        Patch(
            facecolor=site_colors[site], 
            label=f'Site {site}', 
            edgecolor='black',
            linewidth=2,
            alpha=0.9
        )
        for site in unique_sites
    ]
    
    legend = ax.legend(
        handles=legend_elements, 
        title='Site ID Legend', 
        loc='lower right',
        ncol=min(5, len(unique_sites)),      # Max 5 columns
        frameon=True,
        shadow=True,
        fancybox=True,
        fontsize=12,
        title_fontsize=14,
        edgecolor='black',
        facecolor='white',
        framealpha=0.95
    )
    legend.get_frame().set_linewidth(2)
    
    # ----------------------------------------
    # COMPREHENSIVE STATISTICS BOX
    # ----------------------------------------
    total_energy = plot_data['total_energy_daily'].sum()
    avg_per_site = plot_data.groupby('site_id')['total_energy_daily'].sum().mean()
    max_site = plot_data.groupby('site_id')['total_energy_daily'].sum().idxmax()
    max_energy = plot_data.groupby('site_id')['total_energy_daily'].sum().max()
    min_site = plot_data.groupby('site_id')['total_energy_daily'].sum().idxmin()
    min_energy = plot_data.groupby('site_id')['total_energy_daily'].sum().min()
    num_days = len(plot_data['day'].unique())
    
    textstr = f'COMPREHENSIVE STATISTICS\n'
    textstr += f'═══════════════════════════════\n'
    textstr += f'Total Energy: {total_energy:,.0f} kWh\n'
    textstr += f'Avg per Site: {avg_per_site:,.0f} kWh\n'
    textstr += f'\n** TOP PERFORMER **\n'
    textstr += f'   Site {max_site}: {max_energy:,.0f} kWh\n'
    textstr += f'\n** LOWEST CONSUMER **\n'
    textstr += f'   Site {min_site}: {min_energy:,.0f} kWh\n'
    textstr += f'\n** ANALYSIS SCOPE **\n'
    textstr += f'   Sites: {len(unique_sites)}\n'
    textstr += f'   Days: {num_days}\n'
    textstr += f'   Records: {len(plot_data)}'
    
    props = dict(
        boxstyle='round,pad=1.5', 
        facecolor='#fff3e0', 
        alpha=0.98, 
        edgecolor='#f57c00', 
        linewidth=4
    )
    
    ax.text(0.98, 0.98, textstr, 
            transform=ax.transAxes, 
            fontsize=13,
            verticalalignment='top', 
            horizontalalignment='right', 
            bbox=props, 
            family='monospace', 
            fontweight='bold',
            linespacing=1.3)
    
    # ----------------------------------------
    # RANKING TABLE AT BOTTOM
    # ----------------------------------------
    ranking = plot_data.groupby('site_id')['total_energy_daily'].sum().sort_values(ascending=False)
    
    ranking_text = '*** SITE RANKING BY TOTAL ENERGY ***\n'
    ranking_text += '─' * 40 + '\n'
    for rank, (site_id, energy) in enumerate(ranking.items(), 1):
        # Use text markers instead of emojis
        if rank == 1:
            marker = '[1ST]'
        elif rank == 2:
            marker = '[2ND]'
        elif rank == 3:
            marker = '[3RD]'
        else:
            marker = f'[{rank}] '
        
        ranking_text += f'{marker} Site {site_id}: {energy:,.0f} kWh\n'
    
    props2 = dict(
        boxstyle='round,pad=1', 
        facecolor='#e3f2fd', 
        alpha=0.95, 
        edgecolor='#1976d2', 
        linewidth=3
    )
    
    ax.text(0.5, -0.08, ranking_text,
           transform=ax.transAxes, 
           ha='center', 
           va='top',
           fontsize=11,
           family='monospace',
           bbox=props2,
           fontweight='bold')
    
    # ----------------------------------------
    # LIVE DATA FRESHNESS INDICATOR
    # ----------------------------------------
    update_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    ax.text(0.02, 0.98, 
           f'** LIVE STREAMING DATA **\nUpdated: {update_time}\nRecords: {len(plot_data)}',
           transform=ax.transAxes,
           fontsize=10,
           verticalalignment='top',
           bbox=dict(
               boxstyle='round,pad=0.7', 
               facecolor='red', 
               alpha=0.35,
               edgecolor='red',
               linewidth=2
           ))
    
    # ----------------------------------------
    # FINALIZE AND DISPLAY
    # ----------------------------------------
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)  # Make room for ranking table
    plt.savefig('plot2_daily_energy_horizontal.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n" + "="*70)
    print("PLOT 2 CREATED SUCCESSFULLY!")
    print("="*70)
    print(f"Saved as: plot2_daily_energy_horizontal.png")
    print("="*70)
    
    # ----------------------------------------
    # DETAILED CONSOLE OUTPUT
    # ----------------------------------------
    print("\n" + "="*85)
    print("DAILY ENERGY CONSUMPTION ANALYSIS - DETAILED REPORT")
    print("="*85)
    
    print(f"\nSITES ANALYZED: {len(unique_sites)}")
    print(f"   {', '.join([f'Site {s}' for s in sorted(unique_sites)])}")
    
    print(f"\nTIME PERIOD:")
    print(f"   Days Covered: {num_days}")
    print(f"   Date Range: {plot_data['day'].min().date()} to {plot_data['day'].max().date()}")
    print(f"   Total Records: {len(plot_data)}")
    
    print(f"\nPERFORMANCE RANKING:")
    for rank, (site_id, energy) in enumerate(ranking.items(), 1):
        if rank == 1:
            marker = '[1ST]'
        elif rank == 2:
            marker = '[2ND]'
        elif rank == 3:
            marker = '[3RD]'
        else:
            marker = f'[{rank}] '
        
        percentage = (energy / total_energy) * 100
        print(f"   {marker} Site {site_id}: {energy:>15,.0f} kWh ({percentage:>5.1f}%)")
    
    print(f"\nAGGREGATE STATISTICS:")
    print(f"   Total Energy (All Sites): {total_energy:>15,.0f} kWh")
    print(f"   Average per Site:         {avg_per_site:>15,.2f} kWh")
    print(f"   Highest Consumer:         {max_energy:>15,.0f} kWh (Site {max_site})")
    print(f"   Lowest Consumer:          {min_energy:>15,.0f} kWh (Site {min_site})")
    print(f"   Range (Max-Min):          {max_energy - min_energy:>15,.0f} kWh")
    
    print("\n" + "="*85)
    
else:
    print("\n" + "="*70)
    print("CANNOT CREATE PLOT 2 - NO DAILY DATA AVAILABLE")
    print("="*70)

## Shortfall / Excess Energy Analysis

Shortfall or excess is calculated as:

`Predicted Total Energy - Actual Metered Energy`

Positive values indicate over-prediction; negative values indicate under-prediction.

## Shortfall / Excess Analysis Method

Actual meter readings are aggregated by site and day, then joined with predicted daily totals from Kafka. The difference is calculated to identify where the model over- or under-predicts energy demand.

In [ ]:
# ============================================
# CONSUMER VISUALISATION.3: CALCULATE SHORTFALL/EXCESS
# ============================================

if not df_daily.empty and not meters_df.empty:
    print("="*70)
    print("CALCULATING SHORTFALL/EXCESS")
    print("="*70)
    
    # Step 1: Load building information to map building_id → site_id
    print("\nStep 1: Loading building information...")
    try:
        buildings_df = pd.read_csv(BUILDINGS_CSV_PATH)
        building_to_site = buildings_df[['building_id', 'site_id']].drop_duplicates()
        print(f"  Loaded {len(buildings_df)} building records")
        print(f"  Unique buildings: {buildings_df['building_id'].nunique()}")
        print(f"  Unique sites: {buildings_df['site_id'].nunique()}")
    except FileNotFoundError:
        print(f"   {BUILDINGS_CSV_PATH} not found")
        print("  Using only data without site mapping")
        buildings_df = None
    
    # Step 2: Prepare meters data
    print("\nStep 2: Preparing meter data...")
    meters_work = meters_df.copy()
    
    # Convert timestamp to date
    meters_work['ts'] = pd.to_datetime(meters_work['ts'])
    meters_work['day'] = meters_work['ts'].dt.date
    
    # Aggregate by building and day (sum all meter types)
    meters_daily = (meters_work.groupby(['building_id', 'day'])['value']
                    .sum()
                    .reset_index()
                    .rename(columns={'value': 'actual_energy'}))
    
    print(f"  Meters aggregated by building and day")
    print(f"  Shape: {meters_daily.shape}")
    
    # Map buildings to sites
    if buildings_df is not None:
        meters_daily = pd.merge(
            meters_daily,
            building_to_site,
            on='building_id',
            how='left'
        )
        
        # Aggregate by site and day
        meters_daily_by_site = (meters_daily.groupby(['site_id', 'day'])['actual_energy']
                                .sum()
                                .reset_index())
        
        print(f"  Mapped buildings to sites")
        print(f"  Meters by site shape: {meters_daily_by_site.shape}")
    else:
        # Use building_id as site_id
        meters_daily_by_site = meters_daily.copy()
        meters_daily_by_site.rename(columns={'building_id': 'site_id'}, inplace=True)
    
    # Step 3: Prepare predictions data
    print("\nStep 3: Preparing predictions data...")
    df_daily_work = df_daily.copy()
    df_daily_work['day_date'] = pd.to_datetime(df_daily_work['day']).dt.date
    
    predictions_summary = df_daily_work[['site_id', 'day_date', 'total_energy_daily']].copy()
    predictions_summary.columns = ['site_id', 'day', 'predicted_energy']
    
    print(f"  Predictions prepared")
    print(f"  Shape: {predictions_summary.shape}")
    
    # Step 4: Merge predictions with actual
    print("\nStep 4: Merging predictions with actual meter readings...")
    comparison = pd.merge(
        predictions_summary,
        meters_daily_by_site,
        on=['site_id', 'day'],
        how='inner'
    )
    
    print(f"  Merged data")
    print(f"  Shape: {comparison.shape}")
    
    if len(comparison) > 0:
        # Step 5: Calculate differences
        print("\nStep 5: Calculating shortfall/excess...")
        
        # Difference: predicted - actual
        comparison['difference'] = comparison['predicted_energy'] - comparison['actual_energy']
        
        # Percentage error
        comparison['difference_pct'] = (comparison['difference'] / comparison['actual_energy']) * 100
        
        # Absolute difference (for finding best/worst predictions)
        comparison['abs_diff'] = comparison['difference'].abs()
        
        # Convert day back to datetime for plotting
        comparison['day'] = pd.to_datetime(comparison['day'])
        
        print(f"  Calculations complete")
        print(f"\n  Summary Statistics:")
        print(f"    Mean error:     {comparison['difference'].mean():>12,.2f} kWh")
        print(f"    Mean error %:   {comparison['difference_pct'].mean():>8.2f}%")
        print(f"    RMSE:           {(comparison['difference']**2).mean()**0.5:>12,.2f} kWh")
        print(f"    MAE:            {comparison['abs_diff'].mean():>12,.2f} kWh")
        
        print(f"\n  Distribution:")
        over_pred = len(comparison[comparison['difference'] > 0])
        under_pred = len(comparison[comparison['difference'] < 0])
        print(f"    Over-predictions:  {over_pred} ({over_pred/len(comparison)*100:.1f}%)")
        print(f"    Under-predictions: {under_pred} ({under_pred/len(comparison)*100:.1f}%)")
        
        print("\n  Sample comparison data:")
        print(comparison.head(10))
        
        print("\nConsumer visualisation.3 Data Prepared Successfully!")
        
    else:
        print("\nNo matching data between predictions and meters")
        print("   Check:")
        print("   1. Date ranges overlap")
        print("   2. Site IDs match")
        print("   3. Both datasets have data")
        comparison = pd.DataFrame()
        
else:
    print(" Cannot calculate shortfall/excess")
    if df_daily.empty:
        print("   - Daily predictions not available")
    if meters_df.empty:
        print("   - Meters data not available")
    comparison = pd.DataFrame()

## Plot 3: Prediction Shortfall / Excess Visualisation

This plot compares predicted and actual daily energy consumption by site. It highlights prediction error direction and magnitude, helping identify sites where the model performs well or requires improvement.

In [ ]:
# ============================================
# PLOT 3: SHORTFALL/EXCESS ANALYSIS
# Ultimate Visualization with No Overlapping Labels
# ============================================

if not comparison.empty:
    print("Creating Plot 3: Shortfall/Excess Analysis...")
    
    # Create larger figure with grid
    fig = plt.figure(figsize=(24, 12), dpi=120)
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)
    
    # ========================================
    # MAIN PLOT: Large Bar Chart
    # ========================================
    ax_main = fig.add_subplot(gs[0:2, :])
    
    # Sort for better visualization
    comparison_sorted = comparison.sort_values('difference')
    
    # Sophisticated color mapping
    colors = []
    for diff in comparison_sorted['difference']:
        if diff < -5000:
            colors.append('#b71c1c')  # Dark red - severe under
        elif diff < 0:
            colors.append('#ef5350')  # Light red - minor under
        elif diff < 5000:
            colors.append('#66bb6a')  # Light green - minor over
        else:
            colors.append('#2e7d32')  # Dark green - significant over
    
    # Create LARGE bars
    x_pos = range(len(comparison_sorted))
    bars = ax_main.bar(x_pos, comparison_sorted['difference'], 
                       color=colors, alpha=0.85, edgecolor='black', linewidth=2.5,
                       width=0.8)
    
    # Zero line (THICK)
    ax_main.axhline(y=0, color='black', linestyle='-', linewidth=3, alpha=0.8, zorder=5)
    
    # Add value labels on ALL bars
    for i, (bar, val, site, date) in enumerate(zip(
        bars, 
        comparison_sorted['difference'],
        comparison_sorted['site_id'],
        comparison_sorted['day']
    )):
        # Value label
        label_y = val + (abs(val) * 0.05 if val > 0 else -abs(val) * 0.05)
        va = 'bottom' if val > 0 else 'top'
        
        ax_main.text(bar.get_x() + bar.get_width()/2, label_y,
                    f'{val:,.0f}\nkWh',
                    ha='center', va=va,
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.4', 
                             facecolor='white', 
                             alpha=0.9,
                             edgecolor='black',
                             linewidth=1.5))
        
        # Site ID label inside bar (if tall enough)
        if abs(val) > comparison_sorted['difference'].abs().max() * 0.15:
            ax_main.text(bar.get_x() + bar.get_width()/2, val * 0.5,
                        f'Site {site}',
                        ha='center', va='center',
                        fontsize=11, fontweight='bold',
                        color='white',
                        bbox=dict(boxstyle='round,pad=0.4',
                                 facecolor='black',
                                 alpha=0.8,
                                 edgecolor='white',
                                 linewidth=2))
    
    # LARGE labels
    ax_main.set_xlabel('Site ID / Date', fontsize=16, fontweight='bold', labelpad=15)
    ax_main.set_ylabel('Energy Difference (kWh)\nPredicted - Actual', 
                       fontsize=16, fontweight='bold', labelpad=15)
    ax_main.set_title('Energy Prediction Accuracy Analysis\nShortfall (Under-prediction) vs  Excess (Over-prediction)', 
                      fontsize=20, fontweight='bold', pad=25)
    
    # X-axis labels - NO OVERLAP
    if len(comparison_sorted) > 30:
        step = len(comparison_sorted) // 20
        tick_positions = list(range(0, len(comparison_sorted), step))
        tick_labels = [
            f"Site {comparison_sorted.iloc[i]['site_id']}\n{str(comparison_sorted.iloc[i]['day'])[:10]}" 
            if i in tick_positions else '' 
            for i in range(len(comparison_sorted))
        ]
        ax_main.set_xticks(range(len(comparison_sorted)))
        ax_main.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=10, fontweight='bold')
    else:
        x_labels = [f"Site {row['site_id']}\n{str(row['day'])[:10]}" 
                    for _, row in comparison_sorted.iterrows()]
        ax_main.set_xticks(x_pos)
        ax_main.set_xticklabels(x_labels, rotation=60, ha='right', fontsize=9, fontweight='bold')
    
    # Grid
    ax_main.grid(True, alpha=0.4, axis='y', linestyle='--', linewidth=1.2)
    ax_main.set_facecolor('#f8f9fa')
    ax_main.set_axisbelow(True)
    
    # Y-axis formatting
    ax_main.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
    ax_main.tick_params(axis='y', labelsize=13)
    
    # Enhanced legend
    legend_elements = [
        Patch(facecolor='#b71c1c', label=' Severe Under-prediction (< -5000 kWh)', 
              edgecolor='black', linewidth=2),
        Patch(facecolor='#ef5350', label=' Minor Under-prediction (0 to -5000 kWh)', 
              edgecolor='black', linewidth=2),
        Patch(facecolor='#66bb6a', label=' Minor Over-prediction (0 to +5000 kWh)', 
              edgecolor='black', linewidth=2),
        Patch(facecolor='#2e7d32', label=' Significant Over-prediction (> +5000 kWh)', 
              edgecolor='black', linewidth=2)
    ]
    ax_main.legend(handles=legend_elements, loc='upper left', frameon=True, 
                  shadow=True, fancybox=True, fontsize=11, 
                  edgecolor='black', facecolor='white', framealpha=0.95)
    
    # ========================================
    # KPI BOX 1: Accuracy Metrics
    # ========================================
    ax_kpi1 = fig.add_subplot(gs[2, 0])
    ax_kpi1.axis('off')
    
    mean_diff = comparison['difference'].mean()
    mean_pct = comparison['difference_pct'].mean()
    accuracy = 100 - abs(mean_pct)
    rmse = (comparison['difference']**2).mean()**0.5
    mae = comparison['difference'].abs().mean()
    
    kpi1_text = f"""
     ACCURACY METRICS
    ═══════════════════════════
    Mean Error: {mean_diff:,.2f} kWh
    Mean Error %: {mean_pct:.2f}%
    
    Accuracy: {accuracy:.2f}%
    RMSE: {rmse:,.2f} kWh
    MAE: {mae:,.2f} kWh
    """
    ax_kpi1.text(0.5, 0.5, kpi1_text, 
                ha='center', va='center', fontsize=12, family='monospace',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=1.2', facecolor='#e3f2fd', 
                         edgecolor='#1976d2', linewidth=3))
    
    # ========================================
    # KPI BOX 2: Distribution Analysis
    # ========================================
    ax_kpi2 = fig.add_subplot(gs[2, 1])
    ax_kpi2.axis('off')
    
    over_pred = len(comparison[comparison['difference'] > 0])
    under_pred = len(comparison[comparison['difference'] < 0])
    perfect = len(comparison[comparison['difference'] == 0])
    total = len(comparison)
    
    within_10pct = len(comparison[comparison['difference_pct'].abs() < 10])
    within_20pct = len(comparison[comparison['difference_pct'].abs() < 20])
    
    kpi2_text = f"""
     DISTRIBUTION
    ═══════════════════════════
     Over-pred: {over_pred} ({over_pred/total*100:.1f}%)
     Under-pred: {under_pred} ({under_pred/total*100:.1f}%)
     Perfect: {perfect}
    
     Within ±10%: {within_10pct} ({within_10pct/total*100:.1f}%)
     Within ±20%: {within_20pct} ({within_20pct/total*100:.1f}%)
    """
    ax_kpi2.text(0.5, 0.5, kpi2_text,
                ha='center', va='center', fontsize=12, family='monospace',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=1.2', facecolor='#f3e5f5', 
                         edgecolor='#7b1fa2', linewidth=3))
    
    # ========================================
    # KPI BOX 3: Extreme Values
    # ========================================
    ax_kpi3 = fig.add_subplot(gs[2, 2])
    ax_kpi3.axis('off')
    
    max_over = comparison['difference'].max()
    max_over_site = comparison.loc[comparison['difference'].idxmax(), 'site_id']
    max_under = comparison['difference'].min()
    max_under_site = comparison.loc[comparison['difference'].idxmin(), 'site_id']
    
    kpi3_text = f"""
     EXTREMES
    ═══════════════════════════
     Max Over:
       +{max_over:,.0f} kWh
       (Site {max_over_site})
    
     Max Under:
       {max_under:,.0f} kWh
       (Site {max_under_site})
    
     Range: {max_over - max_under:,.0f} kWh
    """
    ax_kpi3.text(0.5, 0.5, kpi3_text,
                ha='center', va='center', fontsize=12, family='monospace',
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=1.2', facecolor='#fff3e0', 
                         edgecolor='#f57c00', linewidth=3))
    
    plt.savefig('plot3_shortfall_excess.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(" Plot 3 created successfully!")
    print("   Saved as: plot3_shortfall_excess.png")
    
    # ========================================
    # DETAILED CONSOLE REPORT
    # ========================================
    print("\n" + "="*85)
    print(" ENERGY PREDICTION ACCURACY ANALYSIS - COMPREHENSIVE REPORT")
    print("="*85)
    
    print(f"\n OVERALL ACCURACY:")
    print(f"   Model Accuracy:           {accuracy:>8.2f}%")
    print(f"   Mean Absolute Error:      {mae:>12,.2f} kWh")
    print(f"   Root Mean Square Error:   {rmse:>12,.2f} kWh")
    print(f"   Mean Error:               {mean_diff:>12,.2f} kWh")
    print(f"   Mean Error Percentage:    {mean_pct:>8.2f}%")
    
    print(f"\n PREDICTION DISTRIBUTION:")
    print(f"   Over-predictions:         {over_pred:>8} ({over_pred/total*100:>5.1f}%)")
    print(f"   Under-predictions:        {under_pred:>8} ({under_pred/total*100:>5.1f}%)")
    print(f"   Perfect predictions:      {perfect:>8}")
    print(f"   Total predictions:        {total:>8}")
    
    print(f"\n ACCURACY BANDS:")
    print(f"   Within ±10%:              {within_10pct:>8} ({within_10pct/total*100:>5.1f}%)")
    print(f"   Within ±20%:              {within_20pct:>8} ({within_20pct/total*100:>5.1f}%)")
    print(f"   Beyond ±20%:              {total-within_20pct:>8} ({(total-within_20pct)/total*100:>5.1f}%)")
    
    print(f"\n EXTREME CASES:")
    print(f"   Max Over-prediction:      {max_over:>12,.0f} kWh (Site {max_over_site})")
    print(f"   Max Under-prediction:     {max_under:>12,.0f} kWh (Site {max_under_site})")
    print(f"   Prediction Range:         {max_over - max_under:>12,.0f} kWh")
    
    print(f"\n BEST PREDICTIONS (Closest to Actual):")
    best_predictions = comparison.nsmallest(5, 'abs_diff')
    for idx, (_, row) in enumerate(best_predictions.iterrows(), 1):
        print(f"   {idx}. Site {row['site_id']} ({row['day'].date()}): {row['difference']:>10,.0f} kWh error")
    
    print(f"\n WORST PREDICTIONS (Furthest from Actual):")
    worst_predictions = comparison.nlargest(5, 'abs_diff')
    for idx, (_, row) in enumerate(worst_predictions.iterrows(), 1):
        print(f"   {idx}. Site {row['site_id']} ({row['day'].date()}): {row['difference']:>10,.0f} kWh error")
    
    print("\n" + "="*85)
    
else:
    print(" Cannot create Plot 3 - no comparison data available")

## Final Summary

This notebook provides the consumer-side visual layer for the building energy streaming project. It converts Kafka outputs into interpretable plots and compares predictions with actual meter readings to support monitoring and model evaluation.

In [ ]:
# ============================================
# CONSUMER VISUALISATION COMPLETION SUMMARY
# ============================================

print("\n" + "="*70)
print(" CONSUMER VISUALISATION COMPLETED SUCCESSFULLY!")
print("="*70)

print("\nConsumer visualisation.1: Loaded Metered Data")
if not meters_df.empty:
    print(f"   - {len(meters_df):,} meter records loaded")
    print(f"   - {meters_df['building_id'].nunique()} unique buildings")
    print(f"   - Date range: {meters_df['ts'].min()} to {meters_df['ts'].max()}")
else:
    print("    No meter data loaded")

print("\nConsumer visualisation.2: Created 2 Visualizations (from 6b and 6c)")
print("   Plot 1: 6-Hour Energy Consumption by Building")
if not df_6h.empty:
    print(f"      - {len(df_6h)} prediction records")
    print(f"      - {df_6h['building_id'].nunique()} buildings analyzed")
    print(f"      - Saved as: plot1_6hour_energy.png")
else:
    print("      No 6-hour data available")

print("   Plot 2: Daily Energy Consumption by Site")
if not df_daily.empty:
    print(f"      - {len(df_daily)} prediction records")
    print(f"      - {df_daily['site_id'].nunique()} sites analyzed")
    print(f"      - Saved as: plot2_daily_energy_horizontal.png")
else:
    print("      No daily data available")

print("\nConsumer visualisation.3: Created Shortfall/Excess Visualization")
print("   Plot 3: Daily Energy Prediction Accuracy by Site")
if not comparison.empty:
    print(f"      - {len(comparison)} comparison records")
    print(f"      - Over-predictions: {len(comparison[comparison['difference'] > 0])}")
    print(f"      - Under-predictions: {len(comparison[comparison['difference'] < 0])}")
    accuracy = 100 - abs(comparison['difference_pct'].mean())
    print(f"      - Model accuracy: {accuracy:.2f}%")
    print(f"      - Saved as: plot3_shortfall_excess.png")
else:
    print("      No comparison data available")

print("\n" + "="*70)
print(" ALL VISUALIZATIONS COMPLETE!")
print("="*70)

print("\n Output Files:")
print("   1. plot1_6hour_energy.png")
print("   2. plot2_daily_energy_horizontal.png")
print("   3. plot3_shortfall_excess.png")

print("\n Ready for:")
print("   - Project submission")
print("   - Demo/interview presentation")
print("   - Further analysis")

print("\n" + "="*70)